In [254]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,davies_bouldin_score

In [255]:
RANDOM_STATE=42
ROOT_DIR = Path().cwd().parent
DATASET_DIR = ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_wholesale_customers.csv"
CSV_FILE_PATH

PosixPath('/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_wholesale_customers.csv')

In [256]:
df = pd.read_csv(CSV_FILE_PATH)
df.head()

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


In [257]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB


In [258]:
X = df.drop(columns=['Channel','Region'])
X.head()

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,12669,9656,7561,214,2674,1338
1,7057,9810,9568,1762,3293,1776
2,6353,8808,7684,2405,3516,7844
3,13265,1196,4221,6404,507,1788
4,22615,5410,7198,3915,1777,5185


In [259]:
def iqr_func(series:pd.Series,k = 1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower,upper

In [260]:
cols_to_cap = X.columns.to_list()
cols_to_cap

['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

In [261]:
X.describe()

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000
mean,12000.297727,5796.265909,7951.277273,3071.931818,2881.493182,1524.870455
std,12647.328865,7380.377175,9503.162829,4854.673333,4767.854448,2820.105937
min,3.000000,55.000000,3.000000,25.000000,3.000000,3.000000
25%,3127.750000,1533.000000,2153.000000,742.250000,256.750000,408.250000
50%,8504.000000,3627.000000,4755.500000,1526.000000,816.500000,965.500000
75%,16933.750000,7190.250000,10655.750000,3554.250000,3922.000000,1820.250000
max,112151.000000,73498.000000,92780.000000,60869.000000,40827.000000,47943.000000


In [262]:
for col in cols_to_cap:
    print(f"Capping col: {col}")
    lower,upper = iqr_func(X[col])
    X[col] = X[col].clip(lower,upper)
    df[col] = df[col].clip(lower,upper)
    print("lower:",lower)
    print("upper:",upper)

Capping col: Fresh
lower: -17581.25
upper: 37642.75
Capping col: Milk
lower: -6952.875
upper: 15676.125
Capping col: Grocery
lower: -10601.125
upper: 23409.875
Capping col: Frozen
lower: -3475.75
upper: 7772.25
Capping col: Detergents_Paper
lower: -5241.125
upper: 9419.875
Capping col: Delicassen
lower: -1709.75
upper: 3938.25


In [263]:
X.describe()

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.000000,440.000000,440.00000,440.000000,440.000000,440.000000
mean,11357.568182,5048.592045,7236.37500,2507.085795,2392.616477,1266.715341
std,10211.542235,4386.377073,6596.53308,2408.297738,2940.794090,1083.069792
min,3.000000,55.000000,3.00000,25.000000,3.000000,3.000000
25%,3127.750000,1533.000000,2153.00000,742.250000,256.750000,408.250000
50%,8504.000000,3627.000000,4755.50000,1526.000000,816.500000,965.500000
75%,16933.750000,7190.250000,10655.75000,3554.250000,3922.000000,1820.250000
max,37642.750000,15676.125000,23409.87500,7772.250000,9419.875000,3938.250000


In [264]:
X

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,12669.00,9656.0,7561.000,214.00,2674.000,1338.00
1,7057.00,9810.0,9568.000,1762.00,3293.000,1776.00
2,6353.00,8808.0,7684.000,2405.00,3516.000,3938.25
3,13265.00,1196.0,4221.000,6404.00,507.000,1788.00
4,22615.00,5410.0,7198.000,3915.00,1777.000,3938.25
...,...,...,...,...,...,...
435,29703.00,12051.0,16027.000,7772.25,182.000,2204.00
436,37642.75,1431.0,764.000,4510.00,93.000,2346.00
437,14531.00,15488.0,23409.875,437.00,9419.875,1867.00
438,10290.00,1981.0,2232.000,1038.00,168.000,2125.00


In [265]:
scaler = StandardScaler()

In [266]:
X_scaled = scaler.fit_transform(X)

In [267]:
X_scaled

array([[ 0.12857261,  1.05158597,  0.04926747, -0.95324427,  0.09579175,
         0.06589216],
       [-0.42162716,  1.08673463,  0.35386453, -0.30973493,  0.30651872,
         0.47075856],
       [-0.49064723,  0.85804007,  0.06793486, -0.04243744,  0.38243489,
         2.46943983],
       ...,
       [ 0.31112285,  2.38267048,  2.45460914, -0.86054235,  2.39229863,
         0.55487464],
       [-0.10466425, -0.70014133, -0.7595007 , -0.61070442, -0.75732904,
         0.7933576 ],
       [-0.84025742, -0.76473271, -0.71730938, -1.01518413, -0.65213577,
        -1.1228252 ]], shape=(440, 6))

In [268]:
for k in range(1,11):
    km = KMeans(n_clusters=k,n_init="auto",random_state=RANDOM_STATE)
    km.fit(X_scaled)
    print(f"K {k} -> SSE {km.inertia_:.2f}")

K 1 -> SSE 2640.00
K 2 -> SSE 1728.19
K 3 -> SSE 1363.45
K 4 -> SSE 1202.41
K 5 -> SSE 1070.15
K 6 -> SSE 964.76
K 7 -> SSE 921.14
K 8 -> SSE 776.63
K 9 -> SSE 726.88
K 10 -> SSE 707.41


In [269]:
kmeans = KMeans(n_clusters=5,n_init="auto",random_state=RANDOM_STATE)
kmeans.fit(X_scaled)
labels = kmeans.predict(X_scaled)

In [270]:
df["Cluster"] = labels.astype(int)
df.head()

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
0,2,3,12669.0,9656.0,7561.0,214.0,2674.0,1338.00,0
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0
2,2,3,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25,0
3,1,3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00,3
4,2,3,22615.0,5410.0,7198.0,3915.0,1777.0,3938.25,3


In [271]:
silhouette = silhouette_score(X_scaled,labels)
davies_bouldin = davies_bouldin_score(X_scaled,labels)

In [272]:
print(f"Silhouette Score : {silhouette:.2f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {davies_bouldin:.2f} (lower is better)")

Silhouette Score : 0.28 (closer to +1 is better)
Davies–Bouldin   : 1.27 (lower is better)


In [273]:
centers_scaled = kmeans.cluster_centers_
centers_scaled

array([[-0.21126584,  0.40733893,  0.28346246, -0.49091611,  0.30213343,
         0.5592714 ],
       [-0.29228998, -0.66142092, -0.61856526, -0.35781944, -0.54923997,
        -0.54786341],
       [ 0.59843262,  1.99868387,  1.56134373,  0.67073137,  1.04442565,
         2.14165517],
       [ 1.07737298, -0.3741859 , -0.49583081,  1.37702303, -0.61602685,
         0.27751899],
       [-0.63143412,  1.30558167,  1.68670558, -0.53821955,  1.83404027,
        -0.26376277]])

In [274]:
centers_original = scaler.inverse_transform(centers_scaled)
centers_original

array([[ 9202.67105263,  6833.30263158,  9104.11842105,  1326.15789474,
         3280.11842105,  1871.75657895],
       [ 8376.23036649,  2150.64921466,  3160.62827225,  1646.32984293,
          779.2513089 ,   674.01570681],
       [17461.54      , 13805.605     , 17524.12      ,  4120.57      ,
         5460.565     ,  3583.64      ],
       [22346.69886364,  3409.13778409,  3969.32954545,  5819.59659091,
          583.06818182,  1566.94602273],
       [ 4916.98333333, 10768.85416667, 18350.13333333,  1212.36666667,
         7780.01875   ,   981.36666667]])

In [275]:
df_centers = pd.DataFrame(centers_original,columns=cols_to_cap)
df_centers.index.name = "Cluster"
df_centers

,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
Cluster,,,,,,
0,9202.671053,6833.302632,9104.118421,1326.157895,3280.118421,1871.756579
1,8376.230366,2150.649215,3160.628272,1646.329843,779.251309,674.015707
2,17461.540000,13805.605000,17524.120000,4120.570000,5460.565000,3583.640000
3,22346.698864,3409.137784,3969.329545,5819.596591,583.068182,1566.946023
4,4916.983333,10768.854167,18350.133333,1212.366667,7780.018750,981.366667


In [276]:
sample_idx = [1,3,4]
sanity = df.loc[sample_idx]
sanity

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0
3,1,3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00,3
4,2,3,22615.0,5410.0,7198.0,3915.0,1777.0,3938.25,3


In [277]:
df.to_csv('clustered_customers.csv')